#MCP Server

In [0]:
%pip install fastapi fastmcp plotly pandas uvicorn nest_asyncio
dbutils.library.restartPython()

In [0]:
"""
Lakehouse Insight MCP Server — Streamable HTTP Edition
------------------------------------------------------
Supports both stdio (local) and Streamable HTTP (remote) transports.
Compatible with VS Code MCP Inspector, Claude Desktop, and Cloud Run.
"""

import os
import pandas as pd
import plotly.express as px
from fastmcp import FastMCP
from pyspark.sql import SparkSession
from fastapi import FastAPI
import uvicorn
import asyncio
from uvicorn import Config, Server

# -----------------------------------------------------------------------------
# Initialize
# -----------------------------------------------------------------------------
mcp = FastMCP(name="Lakehouse Insight MCP Server")
spark = SparkSession.builder.getOrCreate()

# -----------------------------------------------------------------------------
# TOOLS
# -----------------------------------------------------------------------------
@mcp.tool(description="Run a SQL query on Delta Lake and return the results.")
def query_data(sql: str) -> dict:
    """Execute SQL against Databricks Delta and return rows."""
    try:
        df = spark.sql(sql).toPandas()
        return {
            "columns": list(df.columns),
            "rows": df.to_dict(orient="records"),
            "row_count": len(df),
        }
    except Exception as e:
        return {"error": str(e)}

@mcp.tool(description="Inspect Unity Catalog metadata for schemas and tables.")
def query_metadata(catalog: str = "main") -> dict:
    """Return table metadata for the given Unity Catalog."""
    try:
        df = spark.sql(f"SHOW TABLES IN {catalog}")
        return {"tables": df.toPandas().to_dict(orient="records")}
    except Exception as e:
        return {"error": str(e)}

@mcp.tool(description="Render a chart using Plotly (bar or line).")
def render_chart(data: list[dict], x: str, y: str, chart_type: str = "bar") -> dict:
    """Render a chart from query results."""
    try:
        df = pd.DataFrame(data)
        fig = px.bar(df, x=x, y=y) if chart_type == "bar" else px.line(df, x=x, y=y)
        html = fig.to_html(include_plotlyjs="cdn", full_html=False)
        return {"chart_html": html}
    except Exception as e:
        return {"error": str(e)}

# -----------------------------------------------------------------------------
# PROMPTS
# -----------------------------------------------------------------------------
@mcp.prompt(name="summarize_dataset")
def summarize_dataset() -> str:
    return """You are a data analyst.
Summarize this dataset:
- Column names and types
- Missing or skewed values
- Suggested business insights."""

@mcp.prompt(name="compare_tables")
def compare_tables() -> str:
    return """Compare two Delta tables for:
- Schema alignment
- Row count
- Column overlaps and mismatches."""

# -----------------------------------------------------------------------------
# FASTAPI APP (for HTTP transport)
# -----------------------------------------------------------------------------

# 1. Create the mcp_app FIRST
mcp_app = mcp.http_app()

# 2. Create the main app and PASS the mcp_app's lifespan to it.
#    This is the critical step required by FastMCP.
app = FastAPI(lifespan=mcp_app.lifespan)

@app.get("/")
def root():
    return {"status": "Lakehouse Insight MCP Server running"}

# 3. Now, mount the mcp_app
app.mount("/mcp", mcp_app)

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    mode = os.getenv("MCP_TRANSPORT", "http")  # "stdio" or "http"

    if mode == "stdio":
        print("🧩 Starting Lakehouse Insight MCP Server in STDIO mode...")
        mcp.run()
    else:
        port = int(os.getenv("PORT", 5095))
        print(f"🌐 Starting Lakehouse Insight MCP Server on http://0.0.0.0:{port}")

        # Databricks-safe asyncio runner
        config = Config(app=app, host="0.0.0.0", port=port, log_level="info")
        server = Server(config)

        loop = asyncio.get_event_loop()
        loop.create_task(server.serve())
        if not loop.is_running():
            loop.run_forever()




In [0]:
from fastapi.testclient import TestClient

print("--- Running Tests ---")

# You MUST use the 'with' statement for the lifespan to run
client = TestClient(app)

# Test 1: Root endpoint
print("Testing / ...")
resp_root = client.get("/")
print(f"Status: {resp_root.status_code}")
print(f"Body: {resp_root.json()}")
assert resp_root.status_code == 200

# Test 2: MCP capabilities endpoint
print("\nTesting /mcp/capabilities ...")
resp_mcp = client.get("/mcp/capabilities")
print(f"Status: {resp_mcp.status_code}")
# .json() will work now that it's not a 404
print(f"Body: {resp_mcp.json()}")
assert resp_mcp.status_code == 200

print("\n--- Tests Complete ---")

In [0]:
print("--- Isolating and testing mcp_app directly ---")

# Test the MCP app in isolation, without the main 'app'
with TestClient(mcp_app) as mcp_client:
    try:
        resp_iso = mcp_client.get("/capabilities")
        print(f"Isolated Status: {resp_iso.status_code}")
        print(f"Isolated Body: {resp_iso.json()}")
    except Exception as e:
        print(f"Isolated test failed: {e}")

In [0]:
from fastapi.testclient import TestClient
from json import JSONDecodeError

# Use 'with' to ensure the lifespan 'startup' events run
with TestClient(app) as client:
    response = client.get("/")
    try:
        print(response.json())
    except JSONDecodeError:
        print('Response content is not valid JSON or is empty:', response.content)

In [0]:
from fastapi.testclient import TestClient

# The 'with' statement is critical here
with TestClient(app) as client:
    print("Testing /mcp/capabilities...")
    resp = client.get("/mcp/capabilities")
    print("Status:", resp.status_code)
    print("Body:", resp.text)

In [0]:
import fastmcp
print(fastmcp.__version__)


In [0]:
import asyncio

loop = asyncio.get_event_loop()
loop.stop()